# Generate Electricity Outputs

This notebook uses the helper methods in `electricity_plotting.py` to generate electricity capacity maps, electricity generation stacked area charts, and summary CSVs from MACRO `capacity.csv` and `flows.csv` outputs.

## Configure Paths

Edit these paths for a different MACRO run. The defaults point to `results_008/results`.

In [ ]:
from pathlib import Path
import os
import sys

SCENARIO_DIR = Path.cwd()
if not (SCENARIO_DIR / "electricity_plotting.py").exists():
    candidate = Path("31_provinces_1_period_updatedelec_steel_cement_nonprovincial_aluminum_max_renewables_24hours").resolve()
    if candidate.exists():
        SCENARIO_DIR = candidate

if str(SCENARIO_DIR) not in sys.path:
    sys.path.insert(0, str(SCENARIO_DIR))

RESULTS_DIR = SCENARIO_DIR / "results_008" / "results"
CAPACITY_PATH = RESULTS_DIR / "capacity.csv"
FLOWS_PATH = RESULTS_DIR / "flows.csv"
MAP_PATH = SCENARIO_DIR.parent / "distance_co2pipeline" / "gadm36_CHN_1.json"
OUTPUT_DIR = SCENARIO_DIR / "plots" / "electricity_results_008"
CAPACITY_COLUMN = "capacity"
FLOWS_CHUNKSIZE = 100_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".cache" / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(OUTPUT_DIR / ".cache"))

print(f"Scenario directory: {SCENARIO_DIR}")
print(f"Capacity path: {CAPACITY_PATH}")
print(f"Flows path: {FLOWS_PATH}")
print(f"Map path: {MAP_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Flow chunksize: {FLOWS_CHUNKSIZE:,}")

## Import Plotting Helpers

In [ ]:
from electricity_plotting import (
    TECH_COLORS,
    import_plotting_deps,
    load_china_gdf,
    load_macro_results,
    plot_capacity_map_bars,
    plot_capacity_map_pies,
    plot_generation_area,
    summarize_capacity,
    summarize_generation_from_csv,
)

gpd, plt, Wedge = import_plotting_deps()

## Load MACRO Outputs

In [ ]:
capacity_df, _ = load_macro_results(CAPACITY_PATH, FLOWS_PATH, load_flows=False)
gdf = load_china_gdf(gpd, MAP_PATH)

print(f"Capacity rows: {len(capacity_df):,}")
print("Flow data will be streamed from flows.csv during summary generation.")
print(f"Map provinces: {len(gdf):,}")

## Build Summary Tables

In [ ]:
capacity_by_tech, capacity_by_province = summarize_capacity(
    capacity_df,
    capacity_col=CAPACITY_COLUMN,
)
generation_by_tech, generation_by_time = summarize_generation_from_csv(
    FLOWS_PATH,
    chunksize=FLOWS_CHUNKSIZE,
)

if capacity_by_province.empty:
    raise ValueError(f"No electricity capacity rows found in {CAPACITY_PATH}")
if generation_by_time.shape[1] <= 1:
    raise ValueError(f"No electricity generation flow columns found in {FLOWS_PATH}")

print(f"Capacity provinces: {capacity_by_province.shape[0]}")
print(f"Capacity technologies: {capacity_by_province.shape[1]}")
print(f"Generation technologies: {generation_by_time.shape[1] - 1}")

display(capacity_by_province.head())
display(generation_by_time.head())

## Write Summary CSVs

In [ ]:
capacity_csv = OUTPUT_DIR / "electricity_capacity_by_province_technology.csv"
generation_csv = OUTPUT_DIR / "electricity_generation_by_time_technology.csv"

capacity_by_tech.to_csv(capacity_csv, index=False)
generation_by_tech.to_csv(generation_csv, index=False)

print(f"Saved {capacity_csv}")
print(f"Saved {generation_csv}")

## Capacity Stacked Bar Map

In [ ]:
run_name = CAPACITY_PATH.parent.parent.name if CAPACITY_PATH.parent.name == "results" else CAPACITY_PATH.parent.name
capacity_bar_png = OUTPUT_DIR / "electricity_capacity_map_bars.png"

plot_capacity_map_bars(
    capacity_by_province,
    gdf,
    plt,
    capacity_bar_png,
    colors=TECH_COLORS,
    title=f"Electricity Capacity by Province - {run_name}",
    show=True,
)

print(f"Saved {capacity_bar_png}")

## Capacity Pie Map

In [ ]:
capacity_pie_png = OUTPUT_DIR / "electricity_capacity_map_pies.png"

plot_capacity_map_pies(
    capacity_by_province,
    gdf,
    plt,
    Wedge,
    capacity_pie_png,
    colors=TECH_COLORS,
    title=f"Electricity Capacity Mix by Province - {run_name}",
    show=True,
)

print(f"Saved {capacity_pie_png}")

## Generation Stacked Area Chart

In [ ]:
generation_png = OUTPUT_DIR / "electricity_generation_stacked_area.png"

plot_generation_area(
    generation_by_time,
    plt,
    generation_png,
    colors=TECH_COLORS,
    title=f"Electricity Generation by Technology - {run_name}",
    show=True,
)

print(f"Saved {generation_png}")

## Saved Outputs

In [ ]:
saved_outputs = [
    capacity_csv,
    generation_csv,
    capacity_bar_png,
    capacity_pie_png,
    generation_png,
]

for output_path in saved_outputs:
    size_kb = output_path.stat().st_size / 1024
    print(f"{output_path.name}: {size_kb:,.1f} KB")